In [1]:
import numpy as np
import pandas as pd
from sklearn.svm import OneClassSVM
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_curve

cols = [
    "user_id", "doc_id", "inter_stroke_time", "stroke_duration",
    "start_x", "start_y", "stop_x", "stop_y", "end_to_end_dist",
    "mean_resultant_length", "direction_flag", "end_to_end_direction",
    "phone_id", "vel_p20", "vel_p50", "vel_p80", "acc_p20", "acc_p50",
    "acc_p80", "median_vel_last3", "max_dev_e2e", "dev_p20", "dev_p50",
    "dev_p80", "avg_direction", "traj_length", "e2e_over_traj",
    "avg_velocity", "median_acc_first5", "mid_pressure", "mid_area",
    "mid_finger_orient", "finger_orient_change", "phone_orientation",
]

df = pd.read_csv("data/featMat.csv", header=None, names=cols)

print(df.shape)
print(df["user_id"].nunique(), "users")
print(df.groupby("user_id").size().describe())
print(pd.crosstab(df["user_id"], df["doc_id"]))
print(pd.crosstab(df["doc_id"], df["direction_flag"]))
df.head(20)

(21158, 34)
41 users
count      41.000000
mean      516.048780
std       230.006951
min       186.000000
25%       370.000000
50%       439.000000
75%       651.000000
max      1230.000000
dtype: float64
doc_id     1    2    3    4    5    6    7
user_id                                   
1         76  103   77   72   73    0    0
2        278  340  237   64   81  137   93
3         81  135  127  106  100   91  119
4         32   41   25   76   67    0    0
5         50   54  108   56   61   25   59
6         72   86   77  113  106   26   82
7        105  155  126   92  113    0    0
8         45   80   66  113  136   40  138
9         43   58  106   70   66   29   81
10        41   45   68   77   66   21   69
11        31   45   37  116   89   22  105
12        40   59   42  104   97    0    0
13        45   57   56   71   76    0    0
14        73  201  103  175   99    0    0
15       149  137  140  101  192    0    0
16        33   34   36   88   82   23   86
17       120  193  163

,user_id,doc_id,inter_stroke_time,stroke_duration,start_x,start_y,stop_x,stop_y,end_to_end_dist,mean_resultant_length,...,avg_direction,traj_length,e2e_over_traj,avg_velocity,median_acc_first5,mid_pressure,mid_area,mid_finger_orient,finger_orient_change,phone_orientation
0,37,4,0.341,0.256,27.4160,-27.113,12.3980,-32.960,16.1160,0.35736,...,-2.581200,16.1320,0.99898,63.018,-432.240,0.320,0.044444,0.0,0,1
1,37,4,0.913,0.086,21.7710,-30.037,9.8778,-33.867,12.4950,0.26504,...,-2.199400,12.5960,0.99197,146.470,5117.700,0.600,0.088889,0.0,0,1
2,37,4,0.733,0.111,10.8860,-34.169,26.9120,-34.673,16.0340,0.99118,...,0.086288,16.2030,0.98959,145.970,-3563.500,0.480,0.044444,0.0,0,1
3,37,4,0.448,0.094,19.1510,-34.572,12.7000,-36.790,6.8213,0.59647,...,-2.723100,6.9674,0.97902,74.122,-4604.100,0.400,0.044444,0.0,0,1
4,37,4,0.519,0.056,27.7180,-32.758,14.6150,-37.294,13.8660,0.67819,...,-2.717000,13.8710,0.99965,247.690,-21377.000,0.440,0.044444,0.0,0,1
5,37,4,0.290,0.115,30.1370,-30.641,15.8250,-35.479,15.1080,0.79869,...,-2.791400,15.2220,0.99254,132.360,2831.100,0.405,0.044444,0.0,0,1
6,37,4,1.001,0.091,22.3760,-32.355,11.3900,-33.967,11.1040,0.74862,...,-2.956600,11.1380,0.99700,122.390,-4931.100,0.500,0.044444,0.0,0,1
7,37,4,0.287,0.102,31.7500,-31.044,20.7630,-36.286,12.1730,0.57991,...,-2.540400,12.2440,0.99418,120.040,-5321.500,0.390,0.044444,0.0,0,1
8,37,4,1.627,0.103,22.6790,-34.270,10.6840,-35.379,12.0460,0.55388,...,-2.985900,12.1000,0.99547,117.480,-2638.200,0.470,0.044444,0.0,0,1
9,37,4,0.220,0.045,12.0950,-31.952,16.4290,-32.456,4.3633,0.99472,...,-0.033020,4.3996,0.99175,97.770,-14737.000,0.380,0.044444,0.0,0,1


Load Data

In [2]:
LABELS = ["user_id", "doc_id", "phone_id"]
FEATURES = [c for c in cols if c not in LABELS]

n_before = len(df)
df = df.dropna(subset=FEATURES).reset_index(drop=True)
print(f"dropped {n_before - len(df)} rows ({(n_before - len(df)) / n_before:.2%})")

dropped 310 rows (1.47%)


Clean

In [3]:
TESTS = {
    "inter-session (train 1-4, test 5)": ([1, 2, 3, 4], [5]),
    "long-term (train 1-5, test 6-7)":   ([1, 2, 3, 4, 5], [6, 7]),
}

def eer(genuine_scores, impostor_scores):
    y = np.r_[np.ones(len(genuine_scores)), np.zeros(len(impostor_scores))]
    s = np.r_[genuine_scores, impostor_scores]
    fpr, tpr, _ = roc_curve(y, s)
    fnr = 1 - tpr
    i = np.nanargmin(np.abs(fnr - fpr))
    return (fpr[i] + fnr[i]) / 2

Settings and EER function

In [4]:
def score_user(user, train_docs, test_docs, per_direction=False, nu=0.1):
    train = df[(df.user_id == user) & df.doc_id.isin(train_docs)]
    test = df[df.doc_id.isin(test_docs)].copy()
    test["score"] = np.nan
    groups = [None] if not per_direction else sorted(train.direction_flag.unique())
    for g in groups:
        tr = train if g is None else train[train.direction_flag == g]
        if len(tr) < 20:
            continue
        sc = StandardScaler().fit(tr[FEATURES])
        m = OneClassSVM(kernel="rbf", gamma="scale", nu=nu).fit(sc.transform(tr[FEATURES]))
        idx = test.index if g is None else test.index[test.direction_flag == g]
        if len(idx):
            test.loc[idx, "score"] = m.decision_function(sc.transform(test.loc[idx, FEATURES]))
    return test.dropna(subset=["score"])

def window_scores(test, N):
    test = test.sort_index()
    test["w"] = (test.groupby(["user_id", "doc_id"])["score"]
                 .transform(lambda s: s.rolling(N, min_periods=N).mean()))
    return test.dropna(subset=["w"])

def run(train_docs, test_docs, N, per_direction):
    eers = []
    for user in df["user_id"].unique():
        if len(df[(df.user_id == user) & df.doc_id.isin(train_docs)]) < 20:
            continue
        t = window_scores(score_user(user, train_docs, test_docs, per_direction), N)
        gen, imp = t[t.user_id == user]["w"], t[t.user_id != user]["w"]
        if len(gen) and len(imp):
            eers.append(eer(gen, imp))
    return np.median(eers), len(eers)

Scoring functions

In [5]:
import os
os.makedirs("results", exist_ok=True)

eer_rows = []
for per_dir in [False, True]:
    for N in [1, 5, 10]:
        for name, (tr, te) in TESTS.items():
            med, n = run(tr, te, N, per_dir)
            eer_rows.append(dict(per_direction=per_dir, N=N, test=name,
                                 median_eer=round(med, 4), n_users=n))
eer_table = pd.DataFrame(eer_rows)
eer_table.to_csv("results/eer_by_window.csv", index=False)
eer_table

,per_direction,N,test,median_eer,n_users
0,False,1,"inter-session (train 1-4, test 5)",0.2103,40
1,False,1,"long-term (train 1-5, test 6-7)",0.2881,14
2,False,5,"inter-session (train 1-4, test 5)",0.1473,40
3,False,5,"long-term (train 1-5, test 6-7)",0.2419,14
4,False,10,"inter-session (train 1-4, test 5)",0.1100,40
5,False,10,"long-term (train 1-5, test 6-7)",0.2062,14
6,True,1,"inter-session (train 1-4, test 5)",0.1900,40
7,True,1,"long-term (train 1-5, test 6-7)",0.2176,14
8,True,5,"inter-session (train 1-4, test 5)",0.0924,39
9,True,5,"long-term (train 1-5, test 6-7)",0.1597,14


EER experiments

In [6]:
df["stroke_cycle"] = df["inter_stroke_time"] + df["stroke_duration"]

for N in [1, 5, 10]:
    span = (df.groupby(["user_id", "doc_id"])["stroke_cycle"]
              .transform(lambda s: s.rolling(N, min_periods=N).sum())
              .dropna())
    print(f"N={N:2}  median = {span.median():5.1f} s   "
          f"mean = {span.mean():5.1f} s   90th pct = {span.quantile(0.9):5.1f} s")

N= 1  median =   2.2 s   mean =  35.3 s   90th pct =  10.8 s
N= 5  median =  13.5 s   mean = 116.3 s   90th pct =  52.5 s
N=10  median =  28.6 s   mean = 144.2 s   90th pct = 100.5 s


Window duration

In [7]:
def band_table(train_docs, test_docs, N, per_direction):
    parts = []
    for user in df["user_id"].unique():
        if len(df[(df.user_id == user) & df.doc_id.isin(train_docs)]) < 20:
            continue
        s = score_user(user, train_docs, train_docs + test_docs, per_direction)
        w = window_scores(s, N)
        cal = np.sort(w[(w.user_id != user) & w.doc_id.isin(train_docs)]["w"].values)
        tst = w[w.doc_id.isin(test_docs)].copy()
        if len(cal) == 0 or len(tst) == 0:
            continue
        tst["conf"] = 100 * np.searchsorted(cal, tst["w"].values, side="right") / len(cal)
        tst["genuine"] = np.where(tst.user_id == user, "genuine", "impostor")
        tst["band"] = pd.cut(tst["conf"], [-0.1, 50, 90, 100.1],
                             labels=["<50 (terminate)", "50-89 (step-up)", ">=90 (accept)"],
                             right=False)
        parts.append(tst[["genuine", "band"]])
    out = pd.concat(parts)
    counts = pd.crosstab(out["genuine"], out["band"])
    return counts, (100 * counts.div(counts.sum(axis=1), axis=0)).round(1)

for i, (name, (tr, te)) in enumerate(TESTS.items()):
    counts, pct = band_table(tr, te, N=10, per_direction=True)
    counts.to_csv(f"results/bands_{i}.csv")
    print(name)
    print(counts, "\n")
    print(pct, "\n")

inter-session (train 1-4, test 5)
band      <50 (terminate)  50-89 (step-up)  >=90 (accept)
genuine                                                  
genuine                47             1150           1954
impostor            21438            90495          10342 

band      <50 (terminate)  50-89 (step-up)  >=90 (accept)
genuine                                                  
genuine               1.5             36.5           62.0
impostor             17.5             74.0            8.5 

long-term (train 1-5, test 6-7)
band      <50 (terminate)  50-89 (step-up)  >=90 (accept)
genuine                                                  
genuine                22              506           1238
impostor            26494            37666           6548 

band      <50 (terminate)  50-89 (step-up)  >=90 (accept)
genuine                                                  
genuine               1.2             28.7           70.1
impostor             37.5             53.3            9.3 

Confidence bands

In [8]:
df["stroke_cycle"] = df["inter_stroke_time"] + df["stroke_duration"]
df["elapsed"] = df.groupby(["user_id", "doc_id"])["stroke_cycle"].cumsum()

def detection_times(train_docs, test_docs, N, per_direction):
    rows = []
    for user in df["user_id"].unique():
        if len(df[(df.user_id == user) & df.doc_id.isin(train_docs)]) < 20:
            continue
        s = score_user(user, train_docs, train_docs + test_docs, per_direction)
        w = window_scores(s, N)
        cal = np.sort(w[(w.user_id != user) & w.doc_id.isin(train_docs)]["w"].values)
        att = w[(w.user_id != user) & w.doc_id.isin(test_docs)].copy()
        if len(cal) == 0 or len(att) == 0:
            continue
        att["conf"] = 100 * np.searchsorted(cal, att["w"].values, side="right") / len(cal)
        for (imp, doc), g in att.sort_index().groupby(["user_id", "doc_id"]):
            flagged = g[g["conf"] < 90]
            rows.append(flagged["elapsed"].iloc[0] if len(flagged) else np.nan)
    return np.array(rows)

for name, (tr, te) in TESTS.items():
    t = detection_times(tr, te, N=10, per_direction=True)
    det = t[~np.isnan(t)]
    print(f"{name}: {len(t)} attack sessions, "
          f"{100*len(det)/len(t):.1f}% flagged at some point, "
          f"median time to first flag = {np.median(det):.1f} s, "
          f"90th pct = {np.quantile(det, 0.9):.1f} s")

inter-session (train 1-4, test 5): 1562 attack sessions, 98.0% flagged at some point, median time to first flag = 18.8 s, 90th pct = 39.0 s
long-term (train 1-5, test 6-7): 1120 attack sessions, 97.9% flagged at some point, median time to first flag = 33.7 s, 90th pct = 164.7 s


Detection time (impostors)

In [9]:
def first_flag_times(train_docs, test_docs, N, per_direction, genuine):
    rows = []
    for user in df["user_id"].unique():
        if len(df[(df.user_id == user) & df.doc_id.isin(train_docs)]) < 20:
            continue
        s = score_user(user, train_docs, train_docs + test_docs, per_direction)
        w = window_scores(s, N)
        cal = np.sort(w[(w.user_id != user) & w.doc_id.isin(train_docs)]["w"].values)
        who = (w.user_id == user) if genuine else (w.user_id != user)
        att = w[who & w.doc_id.isin(test_docs)].copy()
        if len(cal) == 0 or len(att) == 0:
            continue
        att["conf"] = 100 * np.searchsorted(cal, att["w"].values, side="right") / len(cal)
        for _, g in att.sort_index().groupby(["user_id", "doc_id"]):
            flagged = g[g["conf"] < 90]
            rows.append(flagged["elapsed"].iloc[0] if len(flagged) else np.nan)
    return np.array(rows)

for name, (tr, te) in TESTS.items():
    t = first_flag_times(tr, te, N=10, per_direction=True, genuine=True)
    det = t[~np.isnan(t)]
    print(f"{name} [GENUINE]: {len(t)} sessions, "
          f"{100*len(det)/len(t):.1f}% flagged, "
          f"median first flag = {np.median(det):.1f} s")

inter-session (train 1-4, test 5) [GENUINE]: 39 sessions, 71.8% flagged, median first flag = 28.5 s
long-term (train 1-5, test 6-7) [GENUINE]: 28 sessions, 60.7% flagged, median first flag = 36.3 s


Detection time (genuine)

In [10]:
def flag_times_multi(train_docs, test_docs, N, per_direction, ks, thr=90):
    out = {(k, who): [] for k in ks for who in ("genuine", "impostor")}
    for user in df["user_id"].unique():
        if len(df[(df.user_id == user) & df.doc_id.isin(train_docs)]) < 20:
            continue
        s = score_user(user, train_docs, train_docs + test_docs, per_direction)
        w = window_scores(s, N)
        cal = np.sort(w[(w.user_id != user) & w.doc_id.isin(train_docs)]["w"].values)
        tst = w[w.doc_id.isin(test_docs)].copy()
        if len(cal) == 0 or len(tst) == 0:
            continue
        conf = 100 * np.searchsorted(cal, tst["w"].values, side="right") / len(cal)
        tst["low"] = conf < thr
        tst["who"] = np.where(tst.user_id == user, "genuine", "impostor")
        for (_, _, who), g in tst.sort_index().groupby(["user_id", "doc_id", "who"]):
            low = g["low"].astype(int)
            for k in ks:
                hit = (low.rolling(k).sum() == k).values
                out[(k, who)].append(g["elapsed"].values[np.argmax(hit)] if hit.any() else np.nan)
    return out

KS = [1, 2, 3, 5]
for name, (tr, te) in TESTS.items():
    res = flag_times_multi(tr, te, N=10, per_direction=True, ks=KS)
    print(name)
    for k in KS:
        for who in ("genuine", "impostor"):
            t = np.array(res[(k, who)])
            det = t[~np.isnan(t)]
            med = np.median(det) if len(det) else float("nan")
            print(f"  k={k} {who:8}: {100*len(det)/len(t):5.1f}% flagged, "
                  f"median first flag = {med:5.1f} s")

inter-session (train 1-4, test 5)
  k=1 genuine :  71.8% flagged, median first flag =  28.5 s
  k=1 impostor:  98.0% flagged, median first flag =  18.8 s
  k=2 genuine :  69.2% flagged, median first flag =  33.8 s
  k=2 impostor:  97.5% flagged, median first flag =  19.2 s
  k=3 genuine :  69.2% flagged, median first flag =  34.9 s
  k=3 impostor:  97.4% flagged, median first flag =  22.1 s
  k=5 genuine :  66.7% flagged, median first flag =  40.9 s
  k=5 impostor:  96.7% flagged, median first flag =  27.1 s
long-term (train 1-5, test 6-7)
  k=1 genuine :  60.7% flagged, median first flag =  36.3 s
  k=1 impostor:  97.9% flagged, median first flag =  33.7 s
  k=2 genuine :  57.1% flagged, median first flag =  37.0 s
  k=2 impostor:  96.9% flagged, median first flag =  36.1 s
  k=3 genuine :  57.1% flagged, median first flag =  62.0 s
  k=3 impostor:  96.3% flagged, median first flag =  42.1 s
  k=5 genuine :  53.6% flagged, median first flag =  62.6 s
  k=5 impostor:  95.4% flagged, me

Consecutive-window rule

In [11]:
def flag_times_thr(train_docs, test_docs, N, per_direction, thrs, ks=(1, 3)):
    out = {(t, k, who): [] for t in thrs for k in ks for who in ("genuine", "impostor")}
    for user in df["user_id"].unique():
        if len(df[(df.user_id == user) & df.doc_id.isin(train_docs)]) < 20:
            continue
        s = score_user(user, train_docs, train_docs + test_docs, per_direction)
        w = window_scores(s, N)
        cal = np.sort(w[(w.user_id != user) & w.doc_id.isin(train_docs)]["w"].values)
        tst = w[w.doc_id.isin(test_docs)].copy()
        if len(cal) == 0 or len(tst) == 0:
            continue
        tst["conf"] = 100 * np.searchsorted(cal, tst["w"].values, side="right") / len(cal)
        tst["who"] = np.where(tst.user_id == user, "genuine", "impostor")
        for (_, _, who), g in tst.sort_index().groupby(["user_id", "doc_id", "who"]):
            for t in thrs:
                low = (g["conf"] < t).astype(int)
                for k in ks:
                    hit = (low.rolling(k).sum() == k).values
                    out[(t, k, who)].append(
                        g["elapsed"].values[np.argmax(hit)] if hit.any() else np.nan)
    return out

THRS = [90, 70, 50, 30, 10]
sweep_rows = []
for name, (tr, te) in TESTS.items():
    res = flag_times_thr(tr, te, N=10, per_direction=True, thrs=THRS)
    for t in THRS:
        for k in (1, 3):
            row = dict(test=name, threshold=t, k=k)
            for who in ("genuine", "impostor"):
                a = np.array(res[(t, k, who)])
                det = a[~np.isnan(a)]
                row[f"{who}_sessions"] = len(a)
                row[f"{who}_flagged_pct"] = round(100 * len(det) / len(a), 1)
                row[f"{who}_median_s"] = round(np.median(det), 1) if len(det) else np.nan
            sweep_rows.append(row)
sweep_table = pd.DataFrame(sweep_rows)
sweep_table.to_csv("results/threshold_sweep.csv", index=False)
sweep_table

,test,threshold,k,genuine_sessions,genuine_flagged_pct,genuine_median_s,impostor_sessions,impostor_flagged_pct,impostor_median_s
0,"inter-session (train 1-4, test 5)",90,1,39,71.8,28.5,1562,98.0,18.8
1,"inter-session (train 1-4, test 5)",90,3,39,69.2,34.9,1562,97.4,22.1
2,"inter-session (train 1-4, test 5)",70,1,39,20.5,25.9,1562,67.5,22.5
3,"inter-session (train 1-4, test 5)",70,3,39,15.4,31.3,1562,62.4,26.1
4,"inter-session (train 1-4, test 5)",50,1,39,5.1,57.1,1562,29.3,26.0
5,"inter-session (train 1-4, test 5)",50,3,39,2.6,34.9,1562,26.6,32.4
6,"inter-session (train 1-4, test 5)",30,1,39,2.6,27.1,1562,8.8,26.6
7,"inter-session (train 1-4, test 5)",30,3,39,2.6,34.9,1562,7.4,32.3
8,"inter-session (train 1-4, test 5)",10,1,39,0.0,NaN,1562,2.0,26.7
9,"inter-session (train 1-4, test 5)",10,3,39,0.0,NaN,1562,1.7,29.5


Threshold sweep